# ResNet-18 CIFAR100 NetAttn Subnetwork Test-Only Notebook

用途：从已有 run 的 checkpoint 加载模型权重，只改/固定 NetAttn subnetwork attention，然后只运行 test，不训练。

- `SELECTION_MODE = "manual"`：直接使用 `MANUAL_FIXED_SPEC`。
- `SELECTION_MODE = "topk"`：从 `attention_final.csv` 或 `attention_latest.csv` 读取 attention，选 TopK 路径并生成固定 spec。
- 参数全部集中在第一个 code 单元。


In [9]:
# =========================
# 0. 参数区：只需要改这里
# =========================

# 目标 run 目录：相对当前 notebook 工作目录，或绝对路径都可以
RUN_DIR = r"runs/ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22"

# ckpt 类型："best" -> best_model.pth；"latest" -> checkpoint_latest.pth；"custom" -> 使用 CKPT_PATH
CKPT_TYPE = "best"
CKPT_PATH = None

# subnetwork 选择方式："manual" 或 "topk"
SELECTION_MODE = "topk"

# manual 模式：支持 dict，或 list of (pattern, value) pairs。
# 注意：如果要先清零所有路径，再打开某几条路径，推荐用 list 保证覆盖顺序。
# 例如：[("XXXXXXXX", 0.0), ("11111111", 1.0)]
MANUAL_FIXED_SPEC = [
    ("XXXXXXXX", 0.0),
    ("11111111", 1.0),
]

# topk 模式：从 RUN_DIR/ATTENTION_CSV_NAME 读取 attention，按 weight 降序选 TOPK 条 bits
TOPK = 4
ATTENTION_CSV_NAME = "attention_final.csv"  # 可改成 attention_latest.csv

# attention 总预算；ResNet-18 有 8 个 residual block，因此默认 2^8 = 256
ATTENTION_TOTAL_BUDGET = 1.0

# topk 固定值模式：
# - "even_budget": TopK 平分 ATTENTION_TOTAL_BUDGET
# - "normalize_csv_to_budget": TopK 原 CSV 权重归一化到 ATTENTION_TOTAL_BUDGET
# - "csv_weight": 直接使用 CSV 里的原 weight
# - "manual_value": 每条 TopK 都设置为 TOPK_MANUAL_VALUE
TOPK_VALUE_MODE = "csv_weight"
TOPK_MANUAL_VALUE = 1.0

# topk 模式是否先把全部路径固定为 0，再打开 TopK；通常保持 True
TOPK_BASE_ZERO_ALL = True

# 当全部 attention entry 都被 fixed，且 fixed value 总和不等于 ATTENTION_TOTAL_BUDGET 时，
# 自动把本次模型实际使用的 budget 改成 fixed value 总和。
# 对 TOPK_VALUE_MODE="csv_weight" 且 TOPK_BASE_ZERO_ALL=True 的场景，建议保持 True，
# 这样可以严格保留训练得到的 TopK attention 原值。
AUTO_MATCH_ATTENTION_BUDGET_TO_FIXED_SUM = True

# 是否让配置版 NetAttn 强制 fixed attention 总和匹配 attention_total_budget。
# 如果要保留 TopK 训练得到的原始 csv_weight，并把其它路径置 0，建议 False。
ENFORCE_ATTENTION_TOTAL_BUDGET = False

# topk 是否过滤掉 weight <= 0 的路径；如果 attention 可能为负，通常建议 False，只按排序选
TOPK_ONLY_POSITIVE = False

# 输出目录：None 表示保存到 RUN_DIR/subnet_test_only
OUTPUT_DIR = None

# 测试配置
IS_SERVER = True
FORCE_CPU = False
GPU = 0
SEED = 42
NUM_CLASSES = 100
BATCH_SIZE = 128
WORKERS = 1
USE_AMP_EVAL = True

# 数据增强参数：testloader 不使用训练增强；这里关闭可减少额外依赖与副作用
USE_MIXUP = False
USE_RANDAUGMENT = False
USE_RANDOM_ERASING = False
MIXUP_ALPHA = 0.75
CUTMIX_ALPHA = 0.5
LABEL_SMOOTHING = 0.0
RANDAUGMENT_CONFIG = "rand-m9-n1-mstd0.4-inc1"
RANDOM_ERASING_PROB = 0.0

# 加载 checkpoint 时默认跳过旧 fixed attention buffer，避免覆盖本 notebook 新设置
SKIP_CKPT_FIXED_ATTENTION_BUFFERS = True

# 输出中认为 attention 非零/被选中的阈值
ATTENTION_EPS = 1e-12


## 1. 环境、导入与路径设置

In [10]:
import os
import sys
import json
import random
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm

# notebook 位于 exp31-ConfigNetAttn；models/datasets/utils 在上一级目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    f"cuda:{GPU}" if (torch.cuda.is_available() and not FORCE_CPU) else "cpu"
)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"device      : {device}")
print(f"RUN_DIR     : {RUN_DIR}")


PROJECT_ROOT: d:\Research\Source code\MaxFormer\MyCode
backend     : cupy
device      : cuda:0
RUN_DIR     : runs/ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22


## 2. 数据集加载（只使用 testloader）

In [11]:
from datasets.cifar100_da_dataset import get_dataloaders

trainloader, testloader, classes, _, mixup_fn, criterion_from_data = get_dataloaders(
    is_server=IS_SERVER,
    batch_size=BATCH_SIZE,
    num_workers=WORKERS,
    use_mixup=USE_MIXUP,
    use_randaugment=USE_RANDAUGMENT,
    use_random_erasing=USE_RANDOM_ERASING,
    mixup_alpha=MIXUP_ALPHA,
    cutmix_alpha=CUTMIX_ALPHA,
    label_smoothing=LABEL_SMOOTHING,
    randaugment_config=RANDAUGMENT_CONFIG,
    random_erasing_prob=RANDOM_ERASING_PROB,
)

print(f"_: {_}")
print(f"test batches      : {len(testloader)}")


CIFAR-100 (Data Augmented) 训练集大小: 50000
CIFAR-100 测试集大小: 10000
数据增强配置:
  - RandAugment: False (rand-m9-n1-mstd0.4-inc1)
  - Mixup: False (alpha=0.75)
  - CutMix: False (alpha=0.5)
  - Random Erasing: False (p=0.0)
  - Label Smoothing: 0.0
is_dynamic_dataset: False
test batches      : 79


## 3. 根据参数构造 fixed_pattern_attention_spec

In [12]:
BLOCK_NAMES = [
    "L1.B0",
    "L1.B1",
    "L2.B0",
    "L2.B1",
    "L3.B0",
    "L3.B1",
    "L4.B0",
    "L4.B1",
]
NUM_RESIDUAL_BLOCKS = len(BLOCK_NAMES)
NUM_SUBNETWORKS = 2**NUM_RESIDUAL_BLOCKS


def normalize_bits(value, width=NUM_RESIDUAL_BLOCKS):
    s = str(value).strip().replace(" ", "").replace("_", "")
    if s.lower().startswith("0b"):
        s = s[2:]
    if set(s) <= {"0", "1"} and len(s) <= width:
        return s.zfill(width)
    raise ValueError(f"Invalid bits string: {value}")


def bits_to_index(bits):
    return int(normalize_bits(bits), 2)


def decode_path(bits):
    bits = normalize_bits(bits)
    return " | ".join(
        f"{BLOCK_NAMES[i]}:{'Full' if bits[i] == '1' else 'Skip'}"
        for i in range(NUM_RESIDUAL_BLOCKS)
    )


def resolve_attention_csv_path():
    candidate = Path(RUN_DIR) / ATTENTION_CSV_NAME
    if candidate.exists():
        return candidate
    fallback_names = ["attention_final.csv", "attention_latest.csv"]
    for name in fallback_names:
        fallback = Path(RUN_DIR) / name
        if fallback.exists():
            print(f"Warning: {candidate} not found, fallback to {fallback}")
            return fallback
    raise FileNotFoundError(
        f"Cannot find attention csv in {RUN_DIR}; tried {ATTENTION_CSV_NAME}, attention_final.csv, attention_latest.csv"
    )


def load_attention_table():
    csv_path = resolve_attention_csv_path()
    df = pd.read_csv(csv_path, dtype={"bits": str})
    if "bits" not in df.columns:
        if "subnet_idx" not in df.columns:
            raise ValueError(
                "attention csv must contain either bits or subnet_idx column"
            )
        df["bits"] = df["subnet_idx"].apply(
            lambda x: format(int(x), f"0{NUM_RESIDUAL_BLOCKS}b")
        )
    df["bits"] = df["bits"].apply(normalize_bits)
    if "subnet_idx" not in df.columns:
        df["subnet_idx"] = df["bits"].apply(bits_to_index)
    if "weight" not in df.columns:
        raise ValueError("attention csv must contain weight column")
    df["weight"] = pd.to_numeric(df["weight"], errors="raise")
    df["path"] = df["bits"].apply(decode_path)
    print(f"Loaded attention table: {csv_path}")
    return df


def build_topk_fixed_spec():
    df = load_attention_table()
    work_df = df.copy()
    if TOPK_ONLY_POSITIVE:
        work_df = work_df[work_df["weight"] > 0].copy()
    if len(work_df) == 0:
        raise ValueError("No candidate subnetworks left after filtering")
    k = min(int(TOPK), len(work_df))
    top_df = work_df.sort_values("weight", ascending=False).head(k).copy()

    mode = TOPK_VALUE_MODE.lower()
    if mode == "even_budget":
        values = np.full(k, float(ATTENTION_TOTAL_BUDGET) / k, dtype=float)
    elif mode == "normalize_csv_to_budget":
        raw = top_df["weight"].astype(float).to_numpy()
        raw_sum = float(raw.sum())
        if abs(raw_sum) <= ATTENTION_EPS:
            raise ValueError("TopK csv weights sum to zero; cannot normalize to budget")
        values = raw / raw_sum * float(ATTENTION_TOTAL_BUDGET)
    elif mode == "csv_weight":
        values = top_df["weight"].astype(float).to_numpy()
    elif mode == "manual_value":
        values = np.full(k, float(TOPK_MANUAL_VALUE), dtype=float)
    else:
        raise ValueError(f"Unsupported TOPK_VALUE_MODE: {TOPK_VALUE_MODE}")

    spec = []
    if TOPK_BASE_ZERO_ALL:
        spec.append(("XXXXXXXX", 0.0))
    for bits, value in zip(top_df["bits"].tolist(), values):
        spec.append((bits, float(value)))

    top_df["fixed_value"] = values
    return spec, top_df


def build_manual_fixed_spec():
    if MANUAL_FIXED_SPEC is None:
        raise ValueError(
            "MANUAL_FIXED_SPEC cannot be None when SELECTION_MODE='manual'"
        )
    if isinstance(MANUAL_FIXED_SPEC, dict):
        rows = [
            (str(pattern), float(value)) for pattern, value in MANUAL_FIXED_SPEC.items()
        ]
    else:
        rows = [(str(pattern), float(value)) for pattern, value in MANUAL_FIXED_SPEC]
    df = pd.DataFrame(rows, columns=["pattern", "fixed_value"])
    return MANUAL_FIXED_SPEC, df


selection_mode = SELECTION_MODE.lower()
if selection_mode == "manual":
    fixed_attention_spec, selected_config_df = build_manual_fixed_spec()
elif selection_mode == "topk":
    fixed_attention_spec, selected_config_df = build_topk_fixed_spec()
else:
    raise ValueError(f"Unsupported SELECTION_MODE: {SELECTION_MODE}")


def spec_to_pairs(spec):
    if spec is None:
        return []
    if isinstance(spec, dict):
        return list(spec.items())
    return list(spec)


def normalize_pattern(pattern):
    p = str(pattern).strip().replace(" ", "").replace("_", "").upper()
    if p.startswith("0B"):
        p = p[2:]
    if len(p) != NUM_RESIDUAL_BLOCKS:
        raise ValueError(f"Pattern length must be {NUM_RESIDUAL_BLOCKS}, got {p}")
    if any(ch not in ("0", "1", "X") for ch in p):
        raise ValueError(f"Pattern must contain only 0/1/X, got {p}")
    return p


def pattern_matches(pattern, bits):
    return all(p == "X" or p == b for p, b in zip(pattern, bits))


def summarize_fixed_attention_spec(spec):
    all_bits = [format(i, f"0{NUM_RESIDUAL_BLOCKS}b") for i in range(NUM_SUBNETWORKS)]
    fixed_mask = np.zeros(NUM_SUBNETWORKS, dtype=bool)
    fixed_values = np.zeros(NUM_SUBNETWORKS, dtype=float)

    for pattern_group, value in spec_to_pairs(spec):
        patterns = (
            [pattern_group] if isinstance(pattern_group, str) else list(pattern_group)
        )
        pattern_value = float(value)
        for pattern in patterns:
            p = normalize_pattern(pattern)
            matched = np.array(
                [pattern_matches(p, bits) for bits in all_bits], dtype=bool
            )
            fixed_mask = np.logical_or(fixed_mask, matched)
            fixed_values[matched] = pattern_value

    # 模型中的 fixed_attention_values 跟随 subnetwork_attention_logits 的 dtype，通常是 float32。
    # 如果这里用 float64 CSV 和 Python sum，可能得到 64.4463462，
    # 但模型内部 float32 sum 会是 64.446350097，进而触发 fixed_total > budget 的微小误差报错。
    # 因此用于 budget 校验/传参的 fixed_total 按模型默认 float32 语义计算。
    fixed_values_model_dtype = fixed_values.astype(np.float32)
    fixed_total = float(fixed_values_model_dtype.sum(dtype=np.float32))
    fixed_total_csv_dtype = float(fixed_values.sum(dtype=np.float64))
    all_fixed = bool(fixed_mask.all())
    fixed_count = int(fixed_mask.sum())
    return (
        fixed_mask,
        fixed_values,
        fixed_total,
        fixed_total_csv_dtype,
        all_fixed,
        fixed_count,
    )


(
    FIXED_ATTENTION_MASK,
    FIXED_ATTENTION_VALUES,
    FIXED_ATTENTION_TOTAL,
    FIXED_ATTENTION_TOTAL_CSV_DTYPE,
    ALL_ATTENTION_ENTRIES_FIXED,
    FIXED_ATTENTION_COUNT,
) = summarize_fixed_attention_spec(fixed_attention_spec)

REQUESTED_ATTENTION_TOTAL_BUDGET = float(ATTENTION_TOTAL_BUDGET)
EFFECTIVE_ATTENTION_TOTAL_BUDGET = REQUESTED_ATTENTION_TOTAL_BUDGET

if ENFORCE_ATTENTION_TOTAL_BUDGET:
    if (
        ALL_ATTENTION_ENTRIES_FIXED
        and abs(FIXED_ATTENTION_TOTAL - REQUESTED_ATTENTION_TOTAL_BUDGET) > 1e-6
    ):
        if AUTO_MATCH_ATTENTION_BUDGET_TO_FIXED_SUM:
            if FIXED_ATTENTION_TOTAL <= 0:
                raise ValueError(
                    "All attention entries are fixed, but fixed sum is non-positive. "
                    "Cannot use it as attention_total_budget."
                )
            EFFECTIVE_ATTENTION_TOTAL_BUDGET = FIXED_ATTENTION_TOTAL
            print(
                "[Info] All attention entries are fixed and fixed sum != requested budget."
            )
            print(
                f"       requested ATTENTION_TOTAL_BUDGET = {REQUESTED_ATTENTION_TOTAL_BUDGET:.8f}"
            )
            print(
                f"       fixed attention sum              = {FIXED_ATTENTION_TOTAL:.8f}"
            )
            print(
                f"       EFFECTIVE_ATTENTION_TOTAL_BUDGET = {EFFECTIVE_ATTENTION_TOTAL_BUDGET:.8f}"
            )
            print("       This preserves csv/manual fixed attention values exactly.")
        else:
            raise ValueError(
                "All attention entries are fixed, so fixed sum must equal ATTENTION_TOTAL_BUDGET. "
                f"fixed sum={FIXED_ATTENTION_TOTAL}, requested budget={REQUESTED_ATTENTION_TOTAL_BUDGET}. "
                "Set AUTO_MATCH_ATTENTION_BUDGET_TO_FIXED_SUM=True, or use normalize_csv_to_budget/even_budget."
            )
    elif FIXED_ATTENTION_TOTAL > REQUESTED_ATTENTION_TOTAL_BUDGET + 1e-6:
        raise ValueError(
            f"Fixed attention sum ({FIXED_ATTENTION_TOTAL}) exceeds requested ATTENTION_TOTAL_BUDGET "
            f"({REQUESTED_ATTENTION_TOTAL_BUDGET})."
        )
else:
    print(
        "[Info] ENFORCE_ATTENTION_TOTAL_BUDGET=False; fixed attention values will be used as-is."
    )

print("fixed_attention_spec:")
print(fixed_attention_spec)
print(f"Fixed entries: {FIXED_ATTENTION_COUNT}/{NUM_SUBNETWORKS}")
print(f"Fixed attention sum (model dtype): {FIXED_ATTENTION_TOTAL:.8f}")
print(f"Fixed attention sum (csv/float64): {FIXED_ATTENTION_TOTAL_CSV_DTYPE:.8f}")
print(f"Requested attention budget: {REQUESTED_ATTENTION_TOTAL_BUDGET:.8f}")
print(f"Effective attention budget: {EFFECTIVE_ATTENTION_TOTAL_BUDGET:.8f}")
display(selected_config_df)


Loaded attention table: runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\attention_final.csv
[Info] ENFORCE_ATTENTION_TOTAL_BUDGET=False; fixed attention values will be used as-is.
fixed_attention_spec:
[('XXXXXXXX', 0.0), ('11111111', 0.42762107), ('11101111', 0.26377124), ('01111111', 0.13339111), ('11111100', 0.06367357)]
Fixed entries: 256/256
Fixed attention sum (model dtype): 0.88845694
Fixed attention sum (csv/float64): 0.88845699
Requested attention budget: 1.00000000
Effective attention budget: 1.00000000


,path_idx,bits,weight,path,subnet_idx,fixed_value
36,36,11111111,0.427621,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:F...,255,0.427621
25,25,11101111,0.263771,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:S...,239,0.263771
7,7,01111111,0.133391,L1.B0:Skip | L1.B1:Full | L2.B0:Full | L2.B1:F...,127,0.133391
33,33,11111100,0.063674,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:F...,252,0.063674


## 4. 构建只包含 TopK 路径的 Path NetAttn 模型


In [13]:
from models.ResNet_NetAttn_Path37_CIFAR100 import (
    ResNet_NetAttn_Path37_CIFAR100,
    BasicBlock,
)

selected_path_strings = selected_config_df["bits"].astype(str).tolist()
selected_fixed_values = selected_config_df["fixed_value"].astype(float).to_numpy()
if not selected_path_strings:
    raise ValueError(
        "No selected paths were produced; check TOPK / attention csv settings"
    )
if np.any(selected_fixed_values <= 0):
    raise ValueError(
        "Selected fixed attention values must be positive for logit initialization"
    )

net = ResNet_NetAttn_Path37_CIFAR100(
    block=BasicBlock,
    layers=[2, 2, 2, 2],
    in_channels=3,
    num_classes=NUM_CLASSES,
    zero_init_residual=False,
    path_strings=selected_path_strings,
    attention_total_budget=float(EFFECTIVE_ATTENTION_TOTAL_BUDGET),
)
net.to(device)

eval_criterion = nn.CrossEntropyLoss()


def apply_selected_path_attention():
    values = torch.tensor(
        selected_fixed_values, dtype=net.path_attention_logits.dtype, device=device
    )
    total = values.sum()
    if total <= 0:
        raise ValueError("Selected fixed attention total must be positive")
    # Path model uses attention_total_budget * softmax(path_attention_logits).
    # logits=log(values) and budget=sum(values) exactly reproduces csv_weight.
    with torch.no_grad():
        net.path_attention_logits.copy_(torch.log(values))
    net.attention_total_budget = float(total.detach().cpu().item())
    return float(total.detach().cpu().item())


APPLIED_FIXED_ATTENTION_TOTAL = apply_selected_path_attention()

total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Requested budget    : {REQUESTED_ATTENTION_TOTAL_BUDGET:.6f}")
print(f"Effective budget    : {net.get_attention_total_budget():.6f}")
print(f"Selected Path count : {net.num_paths}")
print("Selected paths      :", selected_path_strings)


Total parameters    : 11,220,136
Trainable parameters: 11,220,136
Requested budget    : 1.000000
Effective budget    : 0.888457
Selected Path count : 4
Selected paths      : ['11111111', '11101111', '01111111', '11111100']


## 5. 加载 checkpoint（跳过原模型 path_attention_logits 后重设 TopK attention）


In [14]:
def resolve_ckpt_path():
    ckpt_type = CKPT_TYPE.lower()
    if ckpt_type in ("best", "best_model"):
        return Path(RUN_DIR) / "best_model.pth"
    if ckpt_type in ("latest", "checkpoint_latest"):
        return Path(RUN_DIR) / "checkpoint_latest.pth"
    if ckpt_type == "custom":
        if CKPT_PATH is None:
            raise ValueError("CKPT_PATH must be set when CKPT_TYPE='custom'")
        return Path(CKPT_PATH)
    raise ValueError(f"Unsupported CKPT_TYPE: {CKPT_TYPE}")


def extract_model_state(loaded_obj):
    if isinstance(loaded_obj, dict) and "model_state_dict" in loaded_obj:
        return loaded_obj["model_state_dict"], loaded_obj
    return loaded_obj, None


def strip_module_prefix(state_dict):
    cleaned = {}
    for key, value in state_dict.items():
        new_key = key[7:] if key.startswith("module.") else key
        cleaned[new_key] = value
    return cleaned


def load_compatible_state_dict(model, state_dict):
    state_dict = strip_module_prefix(state_dict)
    model_state = model.state_dict()
    compatible = {}
    skipped = []

    forced_skip = {"path_attention_logits"}
    if SKIP_CKPT_FIXED_ATTENTION_BUFFERS:
        forced_skip.update(
            {
                "fixed_attention_mask",
                "fixed_attention_values",
                "subnetwork_attention_logits",
            }
        )

    for key, value in state_dict.items():
        if key in forced_skip:
            skipped.append(
                (
                    key,
                    "forced_skip",
                    tuple(value.shape) if hasattr(value, "shape") else None,
                    None,
                )
            )
            continue
        if key in model_state and tuple(value.shape) == tuple(model_state[key].shape):
            compatible[key] = value
        else:
            skipped.append(
                (
                    key,
                    "missing_or_shape_mismatch",
                    tuple(value.shape) if hasattr(value, "shape") else None,
                    tuple(model_state[key].shape)
                    if key in model_state and hasattr(model_state[key], "shape")
                    else None,
                )
            )

    missing, unexpected = model.load_state_dict(compatible, strict=False)
    return compatible, skipped, missing, unexpected


ckpt_path = resolve_ckpt_path()
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

print(f"Loading checkpoint: {ckpt_path}")
loaded_obj = torch.load(ckpt_path, map_location=device, weights_only=False)
model_state_dict, checkpoint_meta = extract_model_state(loaded_obj)
compatible, skipped, missing, unexpected = load_compatible_state_dict(
    net, model_state_dict
)

# Re-apply selected TopK attention after checkpoint loading.
APPLIED_FIXED_ATTENTION_TOTAL = apply_selected_path_attention()

skipped_df = pd.DataFrame(
    skipped, columns=["key", "reason", "ckpt_shape", "model_shape"]
)
print(f"Loaded compatible keys: {len(compatible)}")
print(f"Skipped keys          : {len(skipped_df)}")
print(f"Missing keys          : {len(missing)}")
print(f"Unexpected keys       : {len(unexpected)}")
print(f"Applied attention sum : {APPLIED_FIXED_ATTENTION_TOTAL:.8f}")
if len(skipped_df):
    display(skipped_df.head(30))


Loading checkpoint: runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\best_model.pth
Loaded compatible keys: 122
Skipped keys          : 1
Missing keys          : 1
Unexpected keys       : 0
Applied attention sum : 0.88845700


,key,reason,ckpt_shape,model_shape
0,path_attention_logits,forced_skip,"(37,)",None


## 6. 仅运行 test

In [15]:
def test_only(epoch=0):
    net.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    autocast_enabled = bool(USE_AMP_EVAL and device.type == "cuda")
    with torch.no_grad():
        progress_bar = tqdm(testloader, desc=f"Epoch {epoch} [Test-Only]")
        for inputs, targets in progress_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            with torch.amp.autocast("cuda", enabled=autocast_enabled):
                outputs = net(inputs)
                loss = eval_criterion(outputs, targets)

            test_loss += float(loss.item())
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            all_labels.extend(targets.detach().cpu().numpy().tolist())
            all_preds.extend(predicted.detach().cpu().numpy().tolist())
            progress_bar.set_postfix(
                loss=test_loss / (progress_bar.n + 1), acc=100.0 * correct / total
            )

    return test_loss / len(testloader), 100.0 * correct / total, all_labels, all_preds


test_loss, test_acc, all_labels, all_preds = test_only(epoch=0)
print(f"\nFinal Test Loss    : {test_loss:.6f}")
print(f"Final Test Accuracy: {test_acc:.2f}%")


Epoch 0 [Test-Only]: 100%|██████████| 79/79 [00:06<00:00, 11.32it/s, acc=81.5, loss=1]    


Final Test Loss    : 1.000002
Final Test Accuracy: 81.46%


## 7. 汇总 attention / subnetwork 结果并保存

In [16]:
def make_attention_result_table():
    attn = net.get_attention_weights()
    if attn is None:
        raise RuntimeError("attention weights are not available; run test_only first")
    attn_np = attn.detach().cpu().numpy().astype(float)
    df = pd.DataFrame(
        {
            "path_idx": np.arange(len(attn_np)),
            "bits": net.get_path_strings(),
            "attention": attn_np,
        }
    )
    df["abs_attention"] = df["attention"].abs()
    df["selected"] = df["abs_attention"] > ATTENTION_EPS
    df["path"] = df["bits"].apply(decode_path)
    return df


timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = Path(RUN_DIR).name
out_dir = (
    Path(OUTPUT_DIR) if OUTPUT_DIR is not None else (Path(RUN_DIR) / "subnet_test_only")
)
out_dir.mkdir(parents=True, exist_ok=True)

attention_result_df = make_attention_result_table()
selected_result_df = (
    attention_result_df[attention_result_df["selected"]]
    .sort_values("attention", ascending=False)
    .copy()
)

selection_mode = f"topk_{int(TOPK)}_{TOPK_VALUE_MODE}_path_only"
selected_csv_path = (
    out_dir / f"selected_subnetworks_{selection_mode}_{CKPT_TYPE}_{timestamp}.csv"
)
attention_csv_path = (
    out_dir / f"all_attention_after_test_{selection_mode}_{CKPT_TYPE}_{timestamp}.csv"
)
summary_csv_path = (
    out_dir / f"test_summary_{selection_mode}_{CKPT_TYPE}_{timestamp}.csv"
)
summary_json_path = (
    out_dir / f"test_summary_{selection_mode}_{CKPT_TYPE}_{timestamp}.json"
)
skipped_csv_path = (
    out_dir / f"skipped_ckpt_keys_{selection_mode}_{CKPT_TYPE}_{timestamp}.csv"
)

selected_result_df.to_csv(selected_csv_path, index=False)
attention_result_df.to_csv(attention_csv_path, index=False)
skipped_df.to_csv(skipped_csv_path, index=False)

summary = {
    "timestamp": timestamp,
    "run_dir": str(RUN_DIR),
    "run_name": run_name,
    "ckpt_type": CKPT_TYPE,
    "ckpt_path": str(ckpt_path),
    "selection_mode": SELECTION_MODE,
    "execution_mode": "path_only_topk",
    "topk": int(TOPK),
    "topk_value_mode": TOPK_VALUE_MODE,
    "attention_total_budget": float(net.get_attention_total_budget()),
    "requested_attention_total_budget": float(REQUESTED_ATTENTION_TOTAL_BUDGET),
    "effective_attention_total_budget": float(net.get_attention_total_budget()),
    "fixed_attention_total": float(APPLIED_FIXED_ATTENTION_TOTAL),
    "fixed_attention_total_csv_dtype": float(FIXED_ATTENTION_TOTAL_CSV_DTYPE),
    "fixed_attention_count": int(len(selected_path_strings)),
    "all_attention_entries_fixed": True,
    "auto_match_attention_budget_to_fixed_sum": bool(
        AUTO_MATCH_ATTENTION_BUDGET_TO_FIXED_SUM
    ),
    "enforce_attention_total_budget": False,
    "topk_base_zero_all": bool(TOPK_BASE_ZERO_ALL),
    "topk_only_positive": bool(TOPK_ONLY_POSITIVE),
    "selected_path_count": int(len(selected_path_strings)),
    "num_selected_nonzero_paths": int(len(selected_result_df)),
    "model_num_paths": int(net.num_paths),
    "model_path_strings": selected_path_strings,
    "test_loss": float(test_loss),
    "test_acc": float(test_acc),
    "num_test_samples": int(len(all_labels)),
    "compatible_key_count": int(len(compatible)),
    "skipped_key_count": int(len(skipped_df)),
    "missing_key_count": int(len(missing)),
    "unexpected_key_count": int(len(unexpected)),
    "selected_csv_path": str(selected_csv_path),
    "attention_csv_path": str(attention_csv_path),
    "skipped_csv_path": str(skipped_csv_path),
}

pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=" * 80)
print(f"Saved summary             : {summary_csv_path}")
print(f"Saved summary json        : {summary_json_path}")
print(f"Saved selected subnetworks: {selected_csv_path}")
print(f"Saved all attention       : {attention_csv_path}")
print(f"Saved skipped keys        : {skipped_csv_path}")
print(f"nonzero selected paths: {len(selected_result_df)}")
print("=" * 80)

display(selected_result_df.head(max(20, int(TOPK))))


Saved summary             : runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\subnet_test_only\test_summary_topk_4_csv_weight_path_only_best_2026-07-08_12-56-36.csv
Saved summary json        : runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\subnet_test_only\test_summary_topk_4_csv_weight_path_only_best_2026-07-08_12-56-36.json
Saved selected subnetworks: runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\subnet_test_only\selected_subnetworks_topk_4_csv_weight_path_only_best_2026-07-08_12-56-36.csv
Saved all attention       : runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\subnet_test_only\all_attention_after_test_topk_4_csv_weight_path_only_best_2026-07-08_12-56-36.csv
Saved skipped keys        : runs\ResNet-18-CIFAR100-NetAttn-Path37-nozero-sum=1_2026-07-03_18-14-22\subnet_test_only\skipped_ckpt_keys_topk_4_csv_weight_path_only_best_2026-07-08_12-56-36.csv
nonzero selected paths: 4


,path_idx,bits,attention,abs_attention,selected,path
0,0,11111111,0.427621,0.427621,True,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:F...
1,1,11101111,0.263771,0.263771,True,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:S...
2,2,01111111,0.133391,0.133391,True,L1.B0:Skip | L1.B1:Full | L2.B0:Full | L2.B1:F...
3,3,11111100,0.063674,0.063674,True,L1.B0:Full | L1.B1:Full | L2.B0:Full | L2.B1:F...
